# C7-cnn-transfer — Practice p09 — Solution

The predicted shapes are `(2, 256, 56, 56)`, `(2, 512, 28, 28)`, and `(2, 1024, 14, 14)`. The mid-stage cut has already passed the shape-changing opener of `layer2`.

In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_env_root = os.environ.get("USAAIO_BOOK_ROOT")
if _env_root:
    _root = pathlib.Path(_env_root).resolve()
else:
    _start = pathlib.Path.cwd().resolve()
    _root = next(
        p for p in [_start, *_start.parents]
        if (p / "syllabus.md").is_file() and (p / "curriculum").is_dir()
    )
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; repeat float32 forwards are bit-identical.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32

torch.manual_seed(SEED)
x = torch.randn(2, 3, 224, 224).to(torch.float32)
ch = list(model.children())
trunk_a = nn.Sequential(*ch[:5])
trunk_b = nn.Sequential(*ch[:5], model.layer2[:2])
trunk_c = nn.Sequential(*ch[:7])
with torch.inference_mode():
    shape_a = tuple(trunk_a(x).shape)
    shape_b = tuple(trunk_b(x).shape)
    shape_c = tuple(trunk_c(x).shape)
n_pieces_a = len(trunk_a)
same_object = trunk_a[4] is model.layer1


### Answer check

In [ ]:
assert shape_a == (2, 256, 56, 56)
assert shape_b == (2, 512, 28, 28)
assert shape_c == (2, 1024, 14, 14)
assert n_pieces_a == 5
assert same_object is True
